# 2.2 文本分词

In [1]:
# Listing 2.1 Reading in a short story as text sample into Python
with open("/Users/wangyc/Desktop/projects/build_large_language_model/data/预训练/the-verdict.txt", "r", encoding="utf-8") as f:
        raw_text = f.read()
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20482
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [2]:
# 使用一些简单的示例文本，我们可以使用 re.split 命令，按照以下语法拆分文本中的空白字符：

import re
text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)
print(result)


['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [3]:
# 让我们修改正则表达式，将空白字符（\s）、逗号和句点（[,.]）单独拆分出来：
result = re.split(r'([,.]|\s)', text)
print(result)


['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [4]:
# 一个剩余的小问题是列表仍然包含空白字符。我们可以按如下方式安全地删除这些多余的字符：
result = [item for item in result if item.strip()]
print(result)


['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


关于是否删除空白字符的探讨

在开发一个简单的分词器时，是否将空白字符编码为单独的字符，或者直接将其删除，取决于我们的应用和需求。删除空白字符可以减少内存和计算资源的消耗。然而，如果我们训练的模型对文本的确切结构敏感（例如，Python 代码对缩进和空格非常敏感），那么保留空白字符就很有用。在这里，为了简化和缩短分词化输出，我们选择删除空白字符。稍后，我们将切换到一个包含空白字符的分词化方案。

In [5]:
# 我们上面设计的分词方案在简单的示例文本中表现良好。让我们进一步修改它，使其能够处理其他类型的标点符号，如问号、引号，以及在 Edith Wharton 短篇小说的前 100 个字符中看到的双破折号，还有其他特殊字符：

text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)


['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [6]:
# 现在我们已经有了一个基本的分词器，接下来让我们将其应用于艾迪丝·沃顿的整篇短篇小说：
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed)) # 上述代码的输出是4690，这是小说的token数量（不包含空白字符）。

print(preprocessed[:30])



4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


# 2.3 文本分词将 tokens 转换为token IDs

In [ ]:
# 查看词汇表大小

all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [13]:
# 创建词汇表
vocab = {token:integer for integer,token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i > 60:
        break


('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)
('His', 51)
('How', 52)
('I', 53)
('If', 54)
('In', 55)
('It', 56)
('Jack', 57)
('Jove', 58)
('Just', 59)
('Lord', 60)
('Made', 61)


In [ ]:
# 汇总以上全部代码，构建分词器

# Listing 2.3 Implementing a simple text tokenizer
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab                                                   #A
        self.int_to_str = {i:s for s,i in vocab.items()}                          #B

    def encode(self, text):                                                       #C
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):                                                        #D
        text = " ".join([self.int_to_str[i] for i in ids])

        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)                           #E
        return text


#A 将词汇表作为类属性存储，以方便在 encode 和 decode 方法中访问
#B 创建一个反向词汇表，将token ID 映射回原始的文本token
#C 将输入文本转换为token ID
#D 将token ID 还原为文本
#E 在指定的标点符号前去掉空格
# 在第一步（#D）时，是的，每个 Token 之间都会强制加一个空格。但是第二步（#E）会把标点符号前面的空格删掉。

In [11]:
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)


[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [ ]:
print(tokenizer.decode(ids))


" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [15]:
# 词汇表中没有 ”Hello“
text = "Hello, do you like tea?"
print(tokenizer.encode(text))


KeyError: 'Hello'

# 2.4 添加特殊上下文token

In [16]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer,token in enumerate(all_tokens)}

print(len(vocab.items()))


1132


In [17]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)


('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [18]:
# Listing 2.4 A simple text tokenizer that handles unknown words
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int                    #A
                        else "<|unk|>" for item in preprocessed]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])

        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)                    #B
        return text


#A 用 <|unk|> tokens替换未知词汇
#B 在指定标点符号前替换空格


In [19]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)


Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [20]:
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))


[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


个人思考： 在训练神经网络时，通常会将不同长度的句子或文本批处理为一个 batch 进行并行训练。然而，不同长度的句子需要补齐到同一长度（基于矩阵运算要求形状一致），这时就需要填充 token 来对齐所有序列的长度，使得模型能够有效处理不同长度的输入。掩码其实就是一个标志位，用来告诉大模型哪些位置需要关注，哪些可以忽略，例如考虑以下句子：

句子1："I love NLP."
句子 2："Transformers are powerful."
句子 3："GPT is amazing."
为了将它们放入一个批次，我们需要将它们填充到相同的长度。假设最长句子的长度为 5（token 数量），因此每个句子需要填充到 5 个 token。填充时，GPT 使用 <|endoftext|> 作为填充标记。在输入批次时，我们为每个 token 位置创建一个掩码矩阵，用来标识哪些位置是有效 token（模型应该关注），哪些是填充 token（模型应该忽略）。假设 1 表示有效 token，0 表示填充 token，则掩码矩阵如下：

句子1（掩码矩阵）：[1, 1, 1, 1, 0]
句子2（掩码矩阵）：[1, 1, 1, 1, 0]
句子3（掩码矩阵）：[1, 1, 1, 1, 0]
在这个掩码矩阵中，1 表示模型会关注的 token，0 表示模型会忽略的填充 token。通过这种掩码矩阵，模型知道在计算和训练时哪些 token 是有效内容，哪些 token 是填充部分，无需关注。


此外，用于 GPT 模型的分词器也不使用 <|unk|> 标记来表示词汇表之外的词。相反，GPT 模型采用字节对编码分词器，它将单词分解为子词单元，我们将在下一节中讨论这一点。

# 2.5 字节对编码（Byte pair encoding）

In [21]:
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))


tiktoken version: 0.12.0


In [32]:
tokenizer = tiktoken.get_encoding("gpt2")


In [33]:
text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


In [34]:
strings = tokenizer.decode(integers)
print(strings)


Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


# 2.6 使用滑动窗口进行数据采样

In [38]:
with open("/Users/wangyc/Desktop/projects/build_large_language_model/data/预训练/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))


5147


In [41]:
# 接下来，我们从数据集中移除前50个token以便演示，因为这会在接下来的步骤中产生稍微更有趣的文本段落。
enc_sample = enc_text[50:]

In [42]:
context_size = 4                    #A
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:    {y}")

#A 上下文大小决定输入中包含多少个token


x: [290, 4920, 2241, 287]
y:    [4920, 2241, 287, 257]


In [43]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)


[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [44]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))


 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [ ]:
# 为了实现高效的数据加载器，我们将使用 PyTorch 内置的 Dataset 和 DataLoader 类。

In [55]:
# Listing 2.5 A dataset for batched inputs and targets
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
        def __init__(self, txt, tokenizer, max_length, stride):
                self.input_ids = []
                self.target_ids = []

                token_ids = tokenizer.encode(txt)                                #A

                for i in range(0, len(token_ids) - max_length, stride):          #B
                        input_chunk = token_ids[i:i + max_length]
                        target_chunk = token_ids[i + 1: i + max_length + 1]
                        self.input_ids.append(torch.tensor(input_chunk))
                        self.target_ids.append(torch.tensor(target_chunk))

        def __len__(self):                                                     #C
                return len(self.input_ids)

        def __getitem__(self, idx):                                            #D
                return self.input_ids[idx], self.target_ids[idx]


#A 将整个文本进行分词
#B 使用滑动窗口将书籍分块为最大长度的重叠序列。
#C 返回数据集的总行数
#D 从数据集中返回指定行


In [56]:
# Listing 2.6 A data loader to generate batches with input-with pairs
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                        stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")                       #A
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)      #B
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,                                        #C
        num_workers=0                                               #D
    )

    return dataloader

#A 初始化分词器
#B 创建GPTDatasetV1类
#C drop_last=True会在最后一批次小于指定的batch_size时丢弃该批次，以防止训练期间的损失峰值
#D 用于预处理的CPU进程数量


In [57]:
with open("/Users/wangyc/Desktop/projects/build_large_language_model/data/预训练/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)              #A
first_batch = next(data_iter)
print(first_batch)


#A 将数据加载器转换为 Python 迭代器，以便通过 Python 的内置 next() 函数获取下一个数据条目。


[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [58]:
second_batch = next(data_iter)
print(second_batch)


[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


# 2.7 构建词嵌入层

In [ ]:
input_ids = torch.tensor([2, 3, 5, 1])
vocab_size = 6
output_dim = 3
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim) # 这里是一个大的嵌入表
print(embedding_layer.weight)


Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [63]:
print(embedding_layer(torch.tensor([3])))
print(embedding_layer(input_ids))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


# 2.8 位置编码

在上一节中，我们将token ID 转换为连续的向量表示，即所谓的token嵌入。
原则上，这适合作为 LLM 的输入。然而，LLM的一个小缺点是它们的自注意力机制（将在第3章详细介绍）对序列中token的位置或顺序没有概念。

之前引入的嵌入层的工作方式是，无论token ID 在输入序列中的位置如何，相同的token ID 始终映射到相同的向量表示。

从原则上讲，确定性的、与位置无关的token ID 嵌入对于可重复性是有益的。然而，由于LLM的自注意力机制本身也是与位置无关的，因此向LLM注入额外的位置信息是有帮助的。

In [64]:
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)


In [65]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)


Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [66]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)


torch.Size([8, 4, 256])


对于 GPT 模型所使用的绝对嵌入方法，我们只需创建另一个嵌入层，其维度与 token_embedding_layer 的维度相同：

In [67]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)


torch.Size([4, 256])


In [68]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)


torch.Size([8, 4, 256])


# 2.9 本章小结

- 分词（BPE）-> 词嵌入+位置嵌入 -> 进入解码器
- 可以添加特殊词元（开始符，终止符，填充符等），以增强模型的理解能力
- 嵌入用随机种子初始化，后续通过反向传播调整